# 04. Карты для презентации (дизайн-система maptools)

6 карт в едином макете (2000×1400 px, подложки Esri, линейка, стрелка севера, легенда справа):

1. Четыре версии границ; 2. Возраст застройки; 3. Объекты культурного наследия; 4. Сервисы и зоны пешей доступности; 5. Зелень и промышленность; 6. Плотность жилой застройки (гексагональная сетка).

PNG в `notebooks/maps_out/` (в git не входят), вставляются в `presentations/` без подгонки.

In [1]:
# Настройка: maptools в sys.path, загрузка слоёв, охват карт
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "scripts"))

import maptools  # noqa: E402

PROC = PROJECT / "data/processed"
CACHE = PROJECT / "data/raw/cache"
OUT = PROJECT / "notebooks/maps_out"
OUT.mkdir(exist_ok=True)
CRS_M = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson").set_index("version")
V1, V2, V3, V4 = (versions.loc[v, "geometry"] for v in ["v1", "v2", "v3", "v4"])
building = gpd.read_file(PROC / "building.geojson")
# классификация застройки (как в ноутбуке 03)
LIVING = ["apartments", "residential", "dormitory", "detached", "house", "semidetached_house", "yes"]
building["kind"] = "прочая"
building.loc[building.building.isin(["industrial", "warehouse"]), "kind"] = "промышленно-складская"
building.loc[building.building.isin(LIVING), "kind"] = "жилая"

green = gpd.read_file(PROC / "green.geojson")
services = gpd.read_file(PROC / "services.geojson")
water = gpd.read_file(PROC / "water.geojson")
roads = gpd.read_file(PROC / "roads.geojson")
house_age = gpd.read_file(PROC / "house_age.geojson")
okn = gpd.read_file(PROC / "okn.geojson")

# осевые рек из кэша Overpass (для надёжной отрисовки воды поверх неполного полигона Невы)
import json
from shapely.geometry import LineString
wp = json.load(open(CACHE / "overpass_water_bbox.json", encoding="utf-8"))
axes_neva = [LineString([(g["lon"], g["lat"]) for g in el["geometry"]])
             for el in wp["elements"] if el.get("tags", {}).get("name") == "Нева"]
axes_okhta = [LineString([(g["lon"], g["lat"]) for g in el["geometry"]])
              for el in wp["elements"] if el.get("tags", {}).get("name") == "Охта"]

def draw_water(ax):
    """Вода: полигональный слой + полосы вдоль осевых (полигон Невы из OSM неполный)."""
    water.to_crs(CRS_M).plot(ax=ax, facecolor="#aec6d8", edgecolor="none", alpha=0.85, zorder=1)
    gpd.GeoSeries(axes_neva, crs=4326).to_crs(CRS_M).buffer(260).plot(
        ax=ax, facecolor="#aec6d8", edgecolor="none", alpha=0.85, zorder=1)
    gpd.GeoSeries(axes_okhta, crs=4326).to_crs(CRS_M).buffer(70).plot(
        ax=ax, facecolor="#aec6d8", edgecolor="none", alpha=0.85, zorder=1)

# охват в метрах UTM 36N с запасом 300 м
b = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M)
minx, miny, maxx, maxy = b.total_bounds
EXTENT = (minx - 300, miny - 300, maxx + 300, maxy + 300)
V2_m = gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M)
print("охват карт, м:", [round(v) for v in EXTENT])

охват карт, м: [354073, 6644527, 357751, 6650059]


In [2]:
# Карта 1: четыре версии границ
fig, ax = maptools.map_figure(full=True)

v3_m = gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M)
v2_m = gpd.GeoSeries([V2], crs=4326).to_crs(CRS_M)
v1_m = gpd.GeoSeries([V1], crs=4326).to_crs(CRS_M)
v4_m = gpd.GeoSeries([V4], crs=4326).to_crs(CRS_M)

h = []
v3_m.plot(ax=ax, facecolor="#c7d9ea", edgecolor="none", alpha=0.55, zorder=1)
h.append(Patch(facecolor="#c7d9ea", alpha=0.55, edgecolor="#4a7aa5", label="v3 — вся административная Охта (2 округа)"))
v4_m.boundary.plot(ax=ax, edgecolor="#c98d3a", linewidth=1.8, linestyle="--", zorder=3)
h.append(Line2D([0], [0], color="#c98d3a", lw=1.8, ls="--", label="v4 — охтинский полуостров (1,2 км от рек)"))
v2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=2.2, zorder=4)
h.append(Line2D([0], [0], color="#1f4e79", lw=2.2, label="v2 — МО Малая Охта"))
v1_m.plot(ax=ax, facecolor="none", edgecolor="#8c2d2d", linewidth=2.2, zorder=5)
h.append(Line2D([0], [0], color="#8c2d2d", lw=2.2, label="v1 — историческое ядро (300 м от дореволюционной застройки)"))

# подписи округов
for name, geom, dy in [("Малая Охта", V2, 0), ("Большая Охта", versions.loc["v3", "geometry"].difference(V2), 0)]:
    p = gpd.GeoSeries([geom], crs=4326).to_crs(CRS_M).centroid.iloc[0]
    ax.text(p.x, p.y + dy, name, ha="center", fontsize=11, fontweight="bold", color="#1f4e79",
            path_effects=maptools.HALO, zorder=6)

maptools.finish(fig, ax, EXTENT, h, title="Четыре версии границ района",
                crs=CRS_M, basemap_style="imagery", basemap_alpha=0.85)
maptools.save_map(fig, OUT / "01_boundaries.png")
print("сохранено:", OUT / "01_boundaries.png")

сохранено: /home/mfthesun/Projects/malaya-okhta-strategy/notebooks/maps_out/01_boundaries.png


In [3]:
# Карта 6: плотность жилой застройки по гексагональной сетке 250 м
import numpy as np
from shapely.geometry import Polygon

R = 250  # радиус гексагона, м
living = building[building.kind == "жилая"]
liv_m = living.to_crs(CRS_M)

x0, y0, x1, y1 = EXTENT
dx = R * np.sqrt(3); dy = R * 1.5
cells = []
for i, yc in enumerate(np.arange(y0, y1 + dy, dy)):
    for xc in np.arange(x0, x1 + dx, dx) + (dx / 2 if i % 2 else 0):
        pts = [(xc + R * np.cos(a), yc + R * np.sin(a)) for a in np.linspace(0, 2 * np.pi, 7)[:-1]]
        cells.append(Polygon(pts))
grid = gpd.GeoDataFrame(geometry=cells, crs=CRS_M)
grid = grid[grid.intersects(gpd.GeoSeries([V3], crs=4326).to_crs(CRS_M).iloc[0])]

liv_union = liv_m.geometry.union_all()
grid["footprint"] = grid.geometry.intersection(liv_union).area / 1e4  # га

fig, ax = maptools.map_figure(full=True)
h = []
grid[grid.footprint > 0].plot(column="footprint", cmap="YlOrRd", edgecolor="white", linewidth=0.3,
                              legend=False, ax=ax, zorder=3)
for frac, lab in [(0.1, "< 0,5 га"), (0.5, "0,5–2 га"), (1.0, "> 2 га")]:
    h.append(Patch(facecolor=plt.cm.YlOrRd(frac), edgecolor="white",
                   label=f"жилая застройка в ячейке: {lab}"))
draw_water(ax)
V2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.4, zorder=5)

maptools.finish(fig, ax, EXTENT, h, title="Плотность жилой застройки (гексагоны 250 м)",
                crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "06_density.png")
print("6 готова; ячеек:", len(grid))

6 готова; ячеек: 86


In [4]:
# Карта 5: зелень, промышленность, гаражи — ресурс и барьеры
fig, ax = maptools.map_figure(full=True)
h = []
draw_water(ax)
green[green.kind.isin(["park", "garden"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#7fb069", edgecolor="none", alpha=0.8, zorder=2)
h.append(Patch(facecolor="#7fb069", alpha=0.8, label="парки и сады"))
green[green.kind.isin(["forest", "wood", "grass", "grassland", "meadow", "scrub"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#c5d9a9", edgecolor="none", alpha=0.75, zorder=2)
h.append(Patch(facecolor="#c5d9a9", alpha=0.75, label="лес и травяная зелень"))
green[green.kind == "industrial"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#8a6d3b", edgecolor="none", alpha=0.75, zorder=3)
h.append(Patch(facecolor="#8a6d3b", alpha=0.75, label="промзоны (landuse=industrial; в OSM недоучтены)"))
building[building.building == "industrial"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#6b4f2a", edgecolor="none", alpha=0.9, zorder=4)
h.append(Patch(facecolor="#6b4f2a", alpha=0.9, label="промышленные здания"))
building[building.building == "garages"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#b34a4a", edgecolor="none", alpha=0.9, zorder=4)
h.append(Patch(facecolor="#b34a4a", alpha=0.9, label="гаражные массивы"))
building[building.kind == "жилая"].to_crs(CRS_M).plot(
    ax=ax, facecolor="#d8d4cc", edgecolor="#b5b0a6", linewidth=0.25, alpha=0.9, zorder=3)
V2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.4, zorder=5)

maptools.finish(fig, ax, EXTENT, h, title="Зелень, промышленность, гаражи: ресурс и барьеры",
                crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "05_landuse.png")
print("5 готова")

5 готова


In [5]:
# Карта 4: сетевая доступность (коридоры достижимости по УДС из network_access_fallback.py)
fig, ax = maptools.map_figure(full=True)
h = []
schools = services[services.amenity == "school"]
kinder = services[services.amenity == "kindergarten"]

z_s = gpd.read_file(PROC / "coverage_schools.geojson").to_crs(CRS_M)
z_s.plot(ax=ax, facecolor="#4a7aa5", alpha=0.4, edgecolor="none", zorder=2)
h.append(Patch(facecolor="#4a7aa5", alpha=0.45, label="школы: 500 м по сети от ближайшей"))
z_k = gpd.read_file(PROC / "coverage_kindergartens.geojson").to_crs(CRS_M)
z_k.plot(ax=ax, facecolor="#7fb069", alpha=0.45, edgecolor="none", zorder=3)
h.append(Patch(facecolor="#7fb069", alpha=0.5, label="детсады: 300 м по сети от ближайшего"))

living = building[building.kind == "жилая"]
living.to_crs(CRS_M).plot(ax=ax, facecolor="#d8d4cc", edgecolor="#b5b0a6", linewidth=0.25, zorder=3)
h.append(Patch(facecolor="#d8d4cc", edgecolor="#b5b0a6", label="жилая застройка"))
kinder.to_crs(CRS_M).plot(ax=ax, color="#4f772d", markersize=22, edgecolor="white", linewidth=0.5, zorder=6)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#4f772d", markeredgecolor="white",
                markersize=7, label=f"детсады ({len(kinder)})"))
schools.to_crs(CRS_M).plot(ax=ax, color="#1f4e79", markersize=34, edgecolor="white", linewidth=0.6, zorder=7)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#1f4e79", markeredgecolor="white",
                markersize=9, label=f"школы ({len(schools)})"))
draw_water(ax)
V2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.4, zorder=5)

maptools.finish(fig, ax, EXTENT, h, title="Сетевая доступность: школы и детсады (пешком по УДС)",
                crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "04_services.png")
print("4 готова (сетевые зоны)")

4 готова (сетевые зоны)


In [6]:
# Карта 3: объекты культурного наследия + Ниеншанц
fig, ax = maptools.map_figure(full=True)
h = []
draw_water(ax)
green[green.kind.isin(["park", "forest", "wood"])].to_crs(CRS_M).plot(
    ax=ax, facecolor="#a9c79a", edgecolor="none", alpha=0.55, zorder=2)
h.append(Patch(facecolor="#a9c79a", alpha=0.55, label="парки и зелень"))

fed = okn[okn.cat == "Федерального значения"]
reg = okn[okn.cat == "Регионального значения"]
reg.to_crs(CRS_M).plot(ax=ax, color="#c98d3a", markersize=26, edgecolor="white", linewidth=0.6, zorder=5)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#c98d3a", markeredgecolor="white",
                markersize=8, label=f"ОКН регионального значения ({len(reg)})"))
fed.to_crs(CRS_M).plot(ax=ax, color="#8c2d2d", markersize=42, edgecolor="white", linewidth=0.8, zorder=6)
h.append(Line2D([0], [0], marker="o", color="none", markerfacecolor="#8c2d2d", markeredgecolor="white",
                markersize=10, label=f"ОКН федерального значения ({len(fed)})"))

ny = okn[okn.note.str.contains("Ниеншанц", na=False)]
if len(ny):
    p = ny.to_crs(CRS_M).geometry.iloc[0]
    ax.annotate("Ниеншанц (Охта 1)", xy=(p.x, p.y), xytext=(p.x - 900, p.y + 500), fontsize=9,
                fontweight="bold", color="#8c2d2d", path_effects=maptools.HALO,
                arrowprops=dict(arrowstyle="->", color="#8c2d2d"), zorder=7)

V2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.4, zorder=4)
maptools.finish(fig, ax, EXTENT, h, title="Историко-культурный каркас: ОКН и зелень",
                crs=CRS_M, basemap_style="gray", basemap_alpha=0.95)
maptools.save_map(fig, OUT / "03_heritage.png")
print("3 готова")

3 готова


In [7]:
# Карта 2: возраст застройки (HOITH)
PERIODS = [("до 1917", 1700, 1917, "#8c2d2d"), ("1917–1945", 1917, 1946, "#c98d3a"),
           ("1946–1960", 1946, 1961, "#7fb069"), ("1961–1980", 1961, 1981, "#4a7aa5"),
           ("1981+", 1981, 2027, "#6c5b7b")]

fig, ax = maptools.map_figure(full=True)
h = []
for name, y0, y1, color in PERIODS:
    sel = house_age[(house_age.year >= y0) & (house_age.year < y1)]
    if not len(sel):
        continue
    sel.to_crs(CRS_M).plot(ax=ax, facecolor=color, edgecolor="white", linewidth=0.3, zorder=3)
    h.append(Patch(facecolor=color, edgecolor="white", label=f"{name} ({len(sel)} домов)"))
draw_water(ax)
h.append(Patch(facecolor="#aec6d8", alpha=0.85, label="вода"))
V2_m.boundary.plot(ax=ax, edgecolor="#1f4e79", linewidth=1.6, zorder=5)

maptools.finish(fig, ax, EXTENT, h, title="Возраст жилого фонда (Карта возраста домов)",
                crs=CRS_M, basemap_style="gray", basemap_alpha=0.9)
maptools.save_map(fig, OUT / "02_house_age.png")
print("2 готова; домов с годом:", int(house_age.year.notna().sum()))

2 готова; домов с годом: 1095
